# Criterion: private GPU validation study

Use GPU T4 and Internet. Keep this notebook **private**. This is an exploratory extension after the original CPU test was already evaluated. It does not claim a new blind test. Only the prescribed training and validation patients are embedded. The standalone local browser app remains part of the project.

The 360M target is compared before/after all-layer LoRA, with uniform and task-balanced training objectives. Selection uses validation classification/evidence F1, not test results. GPU benchmark timers synchronize CUDA. Results are measured, never assumed.

In [ ]:
import os, subprocess, sys
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
os.environ["TOKENIZERS_PARALLELISM"]="false"
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU in Settings > Accelerator before running. CPU fallback is disabled.")
print("GPU:",torch.cuda.get_device_name(0),"PyTorch:",torch.__version__)
# Retain Kaggle's CUDA-enabled PyTorch; never install the CPU-only project torch pin.
subprocess.check_call([sys.executable,"-m","pip","install","--quiet",
    "transformers==5.8.0","huggingface-hub==1.14.0","scikit-learn>=1.4,<2","matplotlib>=3.8,<4"])


In [ ]:
import base64, hashlib, io, json, os, zipfile
from pathlib import Path
WORK=Path("/kaggle/working/criterion_gpu")
WORK.mkdir(parents=True,exist_ok=True)
payload=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        target=(WORK/name).resolve()
        if WORK.resolve() not in target.parents: raise ValueError("Unsafe bundle path")
    z.extractall(WORK)
manifest=json.loads((WORK/"source_bundle_manifest.json").read_text())
for name,expected in manifest["files"].items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest()==expected,name
assert not (WORK/"data/processed/test.jsonl").exists()
print("Verified isolated training/validation bundle. No held-out test annotations included.")


In [ ]:
import os, subprocess, sys
os.chdir(WORK)
subprocess.run([sys.executable,"-u","-m","kaggle_study.run"],check=True)


In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(WORK/"criterion_gpu_results.zip")))
print("Keep this notebook private. Download the ZIP after Save Version > Save & Run All finishes.")
